# Module 19: Deployment, Serving & Monitoring

**Snowpark ML Fundamentals — Day 1 Afternoon Session 3**

---

## Learning Objectives

Your model is trained, validated, and registered. Now what?

| Concept | What You'll Learn |
|---------|-------------------|
| Serving patterns | Batch, near-real-time, and real-time inference in Snowflake |
| Feature drift | Detect when production data shifts from training data |
| Feedback loops | When and how to trigger retraining |

## Estimated Time: 30 minutes

---

## 19.1 Batch vs Real-Time Inference Patterns (8 min)

Every ML model needs a serving strategy. Snowflake supports multiple patterns
depending on your latency requirements:

| Pattern | Latency | Snowflake Tool | Use Case |
|---------|---------|----------------|----------|
| **Batch** | Minutes–hours | Stored Procedure + Task | Nightly churn predictions, weekly risk scores |
| **Near-real-time** | Seconds | SQL `MODEL!PREDICT()` | Booking-time pricing, on-demand scoring |
| **Real-time API** | Milliseconds | Snowpark Container Services (SPCS) | Live fraud detection, real-time recommendations |
| **Interactive** | On-demand | Streamlit in Snowflake | Internal dashboards, ad-hoc analysis |

**For NCL:**
- **Batch:** Nightly guest churn/risk predictions scored into a table
- **Near-real-time:** `MODEL!PREDICT()` at booking time for upgrade likelihood
- **Real-time:** SPCS for onboard fraud detection at point-of-sale
- **Interactive:** Streamlit dashboard for revenue management team

In [ ]:
# --- Setup (given — just run this cell) ---
import sys
sys.path.insert(0, '../../../src')

import logging
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("snowflake.snowpark").setLevel(logging.ERROR)
logging.getLogger("snowflake.ml").setLevel(logging.ERROR)

logger = logging.getLogger("module_19")
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(levelname)s:%(name)s:%(message)s"))
    logger.addHandler(handler)
logger.setLevel(logging.INFO)
logger.propagate = False

from snowflake.snowpark import functions as F

from snowpark_fundamentals.session import create_session
from snowpark_fundamentals.data.sample_data import create_customer_churn_dataset
from snowpark_fundamentals.data.loader import split_data
from snowpark_fundamentals.modeling.trainer import train_model, predict
from snowpark_fundamentals.modeling.evaluation import evaluate_binary_classifier
from snowpark_fundamentals.registry.model_registry import (
    get_registry, log_model, delete_model,
    set_model_alias, get_model_by_alias,
    set_model_metrics, get_model_metrics,
)

session = create_session(env_path='../../../.env')
logger.info("Session created.")

In [ ]:
# --- Schema isolation ---
STUDENT_NAME = "YOUR_NAME"  # <-- CHANGE THIS to your name (e.g., "MARIA")

session.sql("USE ROLE MLDS_ROLE_D").collect()
session.sql("USE DATABASE MLDS_D").collect()
session.sql(f"CREATE SCHEMA IF NOT EXISTS {STUDENT_NAME}_ONSITE_TRAINING").collect()
session.sql(f"USE SCHEMA {STUDENT_NAME}_ONSITE_TRAINING").collect()

# Generate data and create a registered model for this session
FEATURE_COLS = [
    'AGE', 'TENURE_MONTHS', 'MONTHLY_CHARGES', 'SUPPORT_TICKETS',
    'USAGE_HOURS_PER_WEEK', 'NUM_DEPENDENTS', 'TOTAL_CHARGES',
]
LABEL_COL = 'CHURNED'

churn_df = create_customer_churn_dataset(session, n_rows=5000)
train_df, test_df = split_data(churn_df, train_ratio=0.8)

model = train_model(train_df, FEATURE_COLS, LABEL_COL, model_type='xgboost',
                    model_params={'n_estimators': 100, 'max_depth': 6})
preds = predict(model, test_df)
metrics = evaluate_binary_classifier(preds, LABEL_COL, 'PREDICTION')

current_db = session.get_current_database().replace('"', '')
current_schema = session.get_current_schema().replace('"', '')
registry = get_registry(session, current_db, current_schema)

MODEL_NAME = f"{STUDENT_NAME}_CHURN_MONITOR"
try:
    delete_model(registry, MODEL_NAME)
except Exception:
    pass

log_model(registry, model.to_xgboost(), MODEL_NAME, 'V1',
          sample_input=test_df.select(FEATURE_COLS).limit(10), metrics=metrics)
set_model_alias(registry, MODEL_NAME, 'V1', 'production')

logger.info(f"Production model ready: {MODEL_NAME} / V1")
logger.info(f"Evaluation metrics (test_df): {metrics}")

In [ ]:
# --- Near-real-time: SQL MODEL!PREDICT() ---
sql_result = session.sql(f"""
    SELECT *
    FROM {current_db}.{current_schema}.CUSTOMER_CHURN_DATA
    LIMIT 5
""")
feature_query = ", ".join(FEATURE_COLS)

sql_preds = session.sql(f"""
    WITH features AS (
        SELECT {feature_query}
        FROM {current_db}.{current_schema}.CUSTOMER_CHURN_DATA
        LIMIT 5
    )
    SELECT *, {current_db}.{current_schema}.{MODEL_NAME}!PREDICT(
        {feature_query}
    ) AS PREDICTION
    FROM features
""")

logger.info("SQL MODEL!PREDICT() \u2014 near-real-time inference:")
sql_preds.show()

---

## 19.2 Feature Drift Detection (12 min)

**The problem:** Your model was trained on data from a specific time period.
If production data distributions shift (seasonality, business changes,
upstream data issues), model predictions degrade.

**The solution:** Monitor feature distributions and alert when they diverge
from training-time baselines.

**Drift score formula:**
```
drift_score = |mean_production - mean_training| / stddev_training
```

A drift score > 2.0 means the feature has shifted by more than 2 standard deviations —
a strong signal that the model's assumptions may no longer hold.

In [ ]:
# TODO 1: Compute training baseline statistics
# Query CUSTOMER_CHURN_DATA to get AVG and STDDEV for 4 features:
#   AGE, TENURE_MONTHS, MONTHLY_CHARGES, SUPPORT_TICKETS
# Hint: Use session.sql() with AVG() and STDDEV() functions
#   Column aliases MUST match feature names:
#     AVG_AGE, STD_AGE, AVG_TENURE_MONTHS, STD_TENURE_MONTHS,
#     AVG_MONTHLY_CHARGES, STD_MONTHLY_CHARGES, AVG_SUPPORT_TICKETS, STD_SUPPORT_TICKETS
#   The print loop below constructs keys from feature names, so aliases must match.
# Note: We use SAMPLE (80 ROWS) to simulate the training subset

baseline_row = session.sql(f"""
    SELECT
        ____,
        ____,
        ____,
        ____
    FROM CUSTOMER_CHURN_DATA
    SAMPLE (80 ROWS)
""").collect()[0]

logger.info("Training baseline statistics:")
for feature in ['AGE', 'TENURE_MONTHS', 'MONTHLY_CHARGES', 'SUPPORT_TICKETS']:
    feat_lower = feature.lower()
    mean_val = float(baseline_row[f'AVG_{feat_lower}'.upper()])
    std_val = float(baseline_row[f'STD_{feat_lower}'.upper()])
    logger.info(f"  {feature}: mean={mean_val:.2f}, std={std_val:.2f}")

In [ ]:
# --- Step 2: Simulate production data 3 months later ---
# Scenario: Post-holiday season — support tickets have increased, tenure patterns shifted
session.sql(f"""
    CREATE OR REPLACE TABLE PRODUCTION_SCORING_DATA AS
    SELECT
        AGE,
        TENURE_MONTHS,
        MONTHLY_CHARGES,
        SUPPORT_TICKETS + 10 AS SUPPORT_TICKETS,
        USAGE_HOURS_PER_WEEK,
        NUM_DEPENDENTS,
        TOTAL_CHARGES,
        CHURNED
    FROM CUSTOMER_CHURN_DATA
    SAMPLE (500 ROWS)
""").collect()

logger.info("Simulated production data with drift:")
logger.info("  - SUPPORT_TICKETS artificially increased by 10 (sharp post-holiday surge)")
logger.info("  - Other features unchanged")
logger.info(f"  - {session.table('PRODUCTION_SCORING_DATA').count()} rows")

In [ ]:
# TODO 2: Compute drift scores for 4 features
# Drift score = |mean_production - mean_training| / stddev_training
# The production statistics query and AGE example are given.
# Fill in the remaining 3 features following the same pattern.
# Hint: Follow the AGE pattern — construct baseline_row keys from feature names
#   e.g., for TENURE_MONTHS: baseline_row['AVG_TENURE_MONTHS'], baseline_row['STD_TENURE_MONTHS']

# Get production statistics
prod_row = session.sql("""
    SELECT
        AVG(AGE) AS AVG_AGE,
        AVG(TENURE_MONTHS) AS AVG_TENURE_MONTHS,
        AVG(MONTHLY_CHARGES) AS AVG_MONTHLY_CHARGES,
        AVG(SUPPORT_TICKETS) AS AVG_SUPPORT_TICKETS
    FROM PRODUCTION_SCORING_DATA
""").collect()[0]

# Compute drift scores
DRIFT_THRESHOLD = 2.0

drift_scores = {
    'AGE': abs(float(prod_row['AVG_AGE']) - float(baseline_row['AVG_AGE'])) / float(baseline_row['STD_AGE']),
    'TENURE_MONTHS': ____,
    'MONTHLY_CHARGES': ____,
    'SUPPORT_TICKETS': ____,
}

logger.info("Feature Drift Report:")
logger.info(f"{'Feature':<20} {'Drift Score':>12} {'Status':>8}")
logger.info("-" * 42)
for feature, score in drift_scores.items():
    status = "ALERT!" if score > DRIFT_THRESHOLD else "OK"
    logger.info(f"{feature:<20} {score:>12.2f} {status:>8}")

In [ ]:
# --- Store drift scores as model metrics for tracking ---
set_model_metrics(registry, MODEL_NAME, 'V1', {
    f'drift_{k.lower()}': round(v, 4) for k, v in drift_scores.items()
})

# Retrieve and display
all_metrics = get_model_metrics(registry, MODEL_NAME, 'V1')
logger.info("Model metrics (training + drift):")
for k, v in sorted(all_metrics.items()):
    logger.info(f"  {k}: {v}")

In [ ]:
# --- Validation: TODO 1 & 2 ---
assert 'baseline_row' in dir(), 'TODO 1: baseline_row must be computed'
assert isinstance(drift_scores, dict), 'TODO 2: drift_scores should be a dict'
assert len(drift_scores) == 4, 'TODO 2: drift_scores should have 4 features'
assert all(isinstance(v, float) for v in drift_scores.values()), \
    'TODO 2: all drift scores should be floats'
# SUPPORT_TICKETS should show drift since we artificially increased it
assert drift_scores['SUPPORT_TICKETS'] > 1.0, \
    "SUPPORT_TICKETS should show significant drift (we increased it)"
logger.info("TODOs 1 & 2: All validations passed!")

In [ ]:
# TODO 3: Create a retraining decision based on drift scores
# Rules:
#   - Retrain if ANY drift_score exceeds DRIFT_THRESHOLD (2.0)
#   - The reason should name the drifted feature(s)
# Return a dict: {"should_retrain": bool, "reason": str, "max_drift": float}
# Hint: Use max() on drift_scores.values() and a list comprehension for drifted features

max_drift = max(drift_scores.values())
drifted_features = ____  # list of feature names where drift > DRIFT_THRESHOLD

retrain_decision = {
    'should_retrain': ____,
    'reason': ____,
    'max_drift': round(max_drift, 4),
}

logger.info(f"Retraining decision: {retrain_decision['should_retrain']}")
logger.info(f"Reason: {retrain_decision['reason']}")
logger.info(f"Max drift score: {retrain_decision['max_drift']}")

In [ ]:
# --- Validation: TODO 3 ---
assert isinstance(retrain_decision, dict), 'TODO 3: retrain_decision should be a dict'
assert 'should_retrain' in retrain_decision, 'TODO 3: must have should_retrain key'
assert isinstance(retrain_decision['should_retrain'], bool), 'TODO 3: should_retrain must be bool'
assert retrain_decision['should_retrain'] is True, \
    "TODO 3: should_retrain should be True (SUPPORT_TICKETS was artificially drifted)"
assert 'SUPPORT_TICKETS' in retrain_decision['reason'], \
    "TODO 3: reason should mention SUPPORT_TICKETS (it has the highest drift)"
logger.info("TODO 3: Validation passed!")

---

## 19.3 Feedback Loops & Retraining Triggers (5 min)

When should you retrain? Four common triggers:

| Trigger | Signal | Automated? |
|---------|--------|------------|
| **Feature drift** | Drift score exceeds threshold (what we just computed) | Yes — stored procedure checks nightly |
| **Prediction shift** | Distribution of predictions changes vs baseline | Yes — compare `GROUP BY prediction` over time |
| **Scheduled** | Calendar-based (weekly, monthly, quarterly) | Yes — Snowflake Task with CRON |
| **Business metric** | Actual outcomes diverge from predictions | Semi — requires ground truth labels |

### The Complete Feedback Loop

```
Train Model
    |
    v
Register in Model Registry (with training metrics + baseline stats)
    |
    v
Deploy via alias ('production')
    |
    v
Score Production Data (batch: SP+Task, real-time: MODEL!PREDICT)
    |
    v
Monitor (feature drift, prediction distribution, business metrics)
    |
    v
Alert if drift_score > threshold OR prediction_shift > tolerance
    |
    v
Retrain (champion/challenger from Module 18)
    |
    v
Promote if challenger wins -> back to Deploy
```

### NCL-Specific Considerations

- **Seasonality:** Guest booking patterns shift dramatically between summer and winter
  cruise seasons. Consider seasonal baseline statistics (summer model vs winter model)
  or quarterly retraining.
- **Event-driven drift:** New ship launches, itinerary changes, or marketing campaigns
  can shift feature distributions suddenly. Drift monitoring catches this.
- **Ground truth delay:** Actual churn/upgrade outcomes may take months to observe.
  Use prediction distribution monitoring as an early warning.

---

## 19.4 Day 1 Recap & Preview

### What We Covered Today

| Session | Key Pattern |
|---------|------------|
| **17: Experimentation** | Model Registry as experiment tracker — version per experiment, version metadata + model tags for discovery |
| **18: Validation** | Holdout + champion/challenger — programmatic promotion with thresholds |
| **19: Monitoring** | Feature drift detection — quantitative alerting before business impact |

### The ML Lifecycle You Now Know

```
Experiment (17) -> Validate (18) -> Deploy -> Monitor (19) -> Retrain -> Validate -> Deploy ...
```

### Tomorrow: Day 2

**Advanced Snowpark Patterns** — UDFs, stored procedures, window functions,
and the Pandas-to-Snowpark migration guide. These are the power tools that
unlock Snowpark's full potential for production workloads.

In [ ]:
# --- Cleanup ---
try:
    session.sql("DROP TABLE IF EXISTS PRODUCTION_SCORING_DATA").collect()
    delete_model(registry, MODEL_NAME)
    logger.info("Cleaned up monitoring resources.")
except Exception as e:
    logger.warning("Cleanup note: %s", e)

In [ ]:
session.close()